# Week 10 — GrowClust3D.jl setup for the PhaseNet/GaMMA catalogue
Reuses Week 6's approach directly: same trace1D mode, same real Chow et al.
(2022) velocity profile (no rebuild needed - it doesn't depend on which
picker/associator produced the catalogue), same aeqd projection. Only the
event list and cross-correlation data are new.

In [ ]:
import os
import pandas as pd

PROJECT_DIR = os.path.expanduser("~/tvz_project")
GC_DIR = os.path.join(PROJECT_DIR, "growclust_phasenet")
GC_IN_DIR = os.path.join(GC_DIR, "in")
GC_OUT_DIR = os.path.join(GC_DIR, "out")
os.makedirs(GC_IN_DIR, exist_ok=True)
os.makedirs(os.path.join(GC_OUT_DIR, "tt"), exist_ok=True)

CATALOG_DIR = os.path.join(PROJECT_DIR, "catalogs")
TARGET_LAT, TARGET_LON = -38.8, 176.0

In [ ]:
# Reuse the EXACT same velocity profile built for the EQTransformer catalogue
# in Week 6 - it's derived from the real Chow et al. (2022) data at Lake
# Taupo and doesn't depend on which picker/associator produced the events.
import shutil

original_vzmodel = os.path.join(PROJECT_DIR, "growclust", "in", "vzmodel.txt")
vzmodel_path = os.path.join(GC_IN_DIR, "vzmodel.txt")
shutil.copy(original_vzmodel, vzmodel_path)

with open(vzmodel_path) as f:
    lines = f.readlines()
print(f"Reused: {vzmodel_path} ({len(lines)} depth levels)")
print("".join(lines[:3]))

In [ ]:
# Event list (evlist_fmt=1): same format as Week 6, evid MUST match dt.cc's
# numbering exactly (event_idx + 1, same convention used when building dt.cc).
filtered = pd.read_csv(os.path.join(CATALOG_DIR, "phasenet_nlloc_quality_filtered.csv"))
filtered["event_idx"] = filtered["file"].str.extract(r"_E(\d+)\.")[0].astype(int)
filtered["origin_dt"] = pd.to_datetime(filtered["origin_time"])

evlist_path = os.path.join(GC_IN_DIR, "evlist.txt")
with open(evlist_path, "w") as f:
    for _, row in filtered.iterrows():
        dt = row["origin_dt"]
        evid = int(row["event_idx"]) + 1
        f.write(
            f"{dt.year:4d} {dt.month:2d} {dt.day:2d} {dt.hour:2d} {dt.minute:2d} "
            f"{dt.second + dt.microsecond / 1e6:6.3f} "
            f"{row['lat']:9.5f} {row['lon']:10.5f} {row['depth_km']:7.3f} "
            f"0.00 {row['horiz_error_km']:6.3f} 0.000 {row['rms']:6.3f} {evid:8d}\n"
        )

print(f"Saved: {evlist_path} ({len(filtered)} events)")

In [ ]:
# Station list: same 60-real-station logic as Week 6 (filtered to stations
# with actual downloaded data, not the full ~209-station regional inventory
# query). Station geometry doesn't depend on the catalogue either, but
# rebuilding directly here keeps this notebook self-contained.
import obspy

INVENTORY_FILE = os.path.join(PROJECT_DIR, "data", "inventory", "NZ_2019-08-01_2019-09-30.xml")
WAVEFORM_DIR = os.path.join(PROJECT_DIR, "data", "waveforms")
real_stations = {d.split(".")[1] for d in os.listdir(WAVEFORM_DIR) if d.startswith("NZ.")}

inventory = obspy.read_inventory(INVENTORY_FILE)
stlist_path = os.path.join(GC_IN_DIR, "stlist2.txt")
written = set()
with open(stlist_path, "w") as f:
    for net in inventory:
        for sta in net:
            if sta.code not in real_stations or sta.code in written:
                continue
            f.write(f"{sta.code:<6s} {sta.latitude:9.5f} {sta.longitude:10.5f} {sta.elevation:7.1f}\n")
            written.add(sta.code)

print(f"Saved: {stlist_path} ({len(written)} stations)")

In [ ]:
# Control file - same trace1D template as Week 6, pointed at the new
# event list and this catalogue's own dt.cc.
dtcc_path = os.path.join(GC_DIR, "dt.cc")

inp_lines = [
    "****  TVZ PhaseNet/GaMMA/NonLinLoc Catalogue  *****",
    "******   GrowClust3D.jl relative relocation   *******",
    "*******************************************",
    "*",
    "*******************************************",
    "*************  Event list  ****************",
    "*******************************************",
    "* evlist_fmt (1 = phase, 2 = GrowClust, 3 = HypoInverse)",
    "1",
    "* fin_evlist (event list file name)",
    "in/evlist.txt",
    "*",
    "*******************************************",
    "************   Station list   *************",
    "*******************************************",
    "* stlist_fmt (1: station name, 2: incl. elev)",
    "2",
    "* fin_stlist (station list file name)",
    "in/stlist2.txt",
    "*",
    "*******************************************",
    "*************   XCOR data   ***************",
    "*******************************************",
    "* xcordat_fmt (1 = text), tdif_fmt (21 = tt2-tt1, 12 = tt1-tt2)",
    "1  12",
    "* fin_xcordat",
    dtcc_path,
    "*",
    "*******************************************",
    "*** Velocity Model / Travel Time Tables ***",
    "*******************************************",
    "* ttabsrc: travel time table source (\"trace\" or \"nllgrid\")",
    "trace",
    "* fin_vzmdl (model name)",
    "in/vzmodel.txt",
    "* fdir_ttab (directory for travel time tables/grids or NONE)",
    "out/tt/",
    "* projection (proj, ellps, lon0, lat0, rotANG, [latP1, latP2])",
    f"aeqd WGS84 {TARGET_LON} {TARGET_LAT} 0.0",
    "******************************************",
    "***** Travel Time Table Parameters  ******",
    "******************************************",
    "* vpvs_factor  rayparam_min",
    "  1.732             0.0",
    "* tt_zmin  tt_zmax  tt_zstep",
    "  -3.0        40.0       1.0",
    "* tt_xmin  tt_xmax  tt_xstep",
    "   0.0       200.0      1.0",
    "*",
    "******************************************",
    "***** GrowClust Algorithm Parameters *****",
    "******************************************",
    "* rmin  delmax rmsmax",
    "   0.6    80    0.2",
    "* rpsavgmin, rmincut  ngoodmin   iponly",
    "    0          0         0        0",
    "*",
    "******************************************",
    "************ Output files ****************",
    "******************************************",
    "* nboot  nbranch_min",
    "   0         2",
    "* fout_cat (relocated catalog)",
    "out/phasenet.relocated.cat",
    "* fout_clust (relocated cluster file)",
    "out/phasenet.relocated.clust",
    "* fout_log (program log)",
    "out/phasenet.relocated.log",
    "* fout_boot (bootstrap distribution)",
    "NONE",
    "******************************************",
    "******************************************",
]

inp_path = os.path.join(GC_DIR, "growclust_phasenet.inp")
with open(inp_path, "w") as f:
    f.write("\n".join(inp_lines) + "\n")

print(f"Saved: {inp_path}")
print("\n".join(inp_lines[:15]))